# Week 07 — Home exercise 4: First look at the emissions data

**Solution proposal.**

The standard set of questions to ask a table you have never seen before, asked in order.

In [1]:
import pandas as pd

co2 = pd.read_csv("../data/co2_emissions.csv")

## 1. How big is it?

In [2]:
print(co2.shape)

(6240, 11)


6 240 rows and 11 columns.

## 2. What types are the columns?

In [3]:
co2.dtypes

country              str
code                 str
year               int64
co2_total        float64
population         int64
urban            float64
gdp_pc           float64
electricity      float64
agriculture      float64
nat_resources    float64
renew_energy     float64
dtype: object

Two are worth a second look.

`population` is `int64`, which is what you would expect of a count — and it is only an integer
because not a single value is missing. Every other numeric column is `float64`, and for most of them
that is because they genuinely have decimals. But `co2_total` would be a float even if every value
were whole, because 336 of them are missing and missing is `NaN`.

The type is telling you about the holes, not about the numbers.

## 3. How many entities, and which years?

In [4]:
print("entities:", co2["country"].nunique())
print("years:   ", co2["year"].min(), "to", co2["year"].max())
print("rows per entity:", co2["country"].value_counts().unique())

entities: 260
years:    2000 to 2023
rows per entity: [24]


260 entities, 24 years, and every entity has exactly 24 rows. A **balanced panel** — unusually
tidy, and worth knowing before you start, because it means a missing value is a hole in the data
rather than a row that was never collected.

## 4. Where is it missing?

In [5]:
co2.isna().sum()

country            0
code               0
year               0
co2_total        336
population         0
urban              0
gdp_pc           166
electricity       73
agriculture      206
nat_resources    759
renew_energy     635
dtype: int64

`country`, `code`, `year`, `population` and `urban` have none. The rest do, and
`nat_resources` and `renew_energy` have a lot.

## 5. The five largest emitters in 2023

In [6]:
year_2023 = co2[co2["year"] == 2023]

year_2023.sort_values("co2_total", ascending=False).head(5)[["country", "code", "co2_total"]]

,country,code,co2_total
6167,World,WLD,39112.6888
2519,IDA & IBRD total,IBT,26334.9494
2495,IBRD only,IBD,25256.2848
3407,Low & middle income,LMY,23925.6677
3815,Middle income,MIC,23703.6333


**Not one of them is a country.** `World`, `IDA & IBRD total`, `IBRD only`,
`Low & middle income` and `Middle income` are all World Bank groupings, and they overlap with each
other as well as containing the countries below them. Sum this column and you count China several
times.

This is the single most important thing to know about this dataset, and nothing in the file announces
it. The `country_info` file says which entities are aggregates — bringing the two together is next
week's work.

## 6. Norway, last five years

In [7]:
norway = co2[co2["country"] == "Norway"]

norway.tail(5)[["year", "co2_total", "gdp_pc"]]

,year,co2_total,gdp_pc
4291,2019,45.3466,79329.31
4292,2020,43.7079,71057.59
4293,2021,44.4278,96442.56
4294,2022,43.5471,113122.13
4295,2023,41.6042,90984.41


## 7. Rich and renewable

In [8]:
year_2020 = co2[co2["year"] == 2020]

rich_green = year_2020[(year_2020["gdp_pc"] > 50000) & (year_2020["renew_energy"] > 40)]

print("rows:", len(rich_green))
rich_green[["country", "gdp_pc", "renew_energy"]]

rows: 4


,country,gdp_pc,renew_energy
2612,Iceland,60127.94,82.9
3356,Liechtenstein,164671.09,55.2
4292,Norway,71057.59,60.9
5516,Sweden,52568.57,57.8


Now the same query for 2023.

In [9]:
rich_green_2023 = year_2023[(year_2023["gdp_pc"] > 50000) & (year_2023["renew_energy"] > 40)]

print("rows:", len(rich_green_2023))

rows: 0


Zero — and **not** because the world changed. `renew_energy` is missing for all 260 entities
in 2023, because the series is published with a lag. A comparison against `NaN` is `False`, so every
row fails the test.

Question 4 has the reason in it: the count of missing values told you this column was thin, and asking
*where* it was thin would have told you exactly which years to avoid.

## Things worth noticing

**Question 7 is the one to remember.** A filter on a column with missing values does not warn you and
does not error: `NaN > 40` is `False`, so those rows are quietly excluded, and an empty result looks
exactly like a true finding of "no countries match". If you had asked "which rich countries went
green?" and reported the answer as none, you would have been wrong, and nothing in the output would
have said so.

The habit that catches it: before filtering on a column, check how much of it exists.
`co2["renew_energy"].isna().sum()` takes two seconds.

**Question 5 is the other one.** The largest values in a dataset are very often not observations at
all but totals shipped in the same file, and sorting by the biggest number is the fastest way to find
them. Doing that once, on any new dataset, is worth the ten seconds it costs.

### What this analysis does NOT do

- It never removes the aggregates, so every number computed over all entities in this file is wrong to
  some degree. The mean emissions figure in particular is meaningless: it averages Tuvalu with World.
- It reports Norway's numbers without checking that "Norway" appears exactly once per year. It does
  here, but that is a fact about this file, not a guarantee — and a country appearing twice under two
  spellings would produce two rows per year with no warning.
- "GDP per capita above 50 000" is in current US dollars and is not adjusted for inflation across the
  24 years, so the same threshold means something different in 2000 and in 2023. That is a property of
  the indicator, not a bug, but it makes comparisons over time misleading.